# Lab 01: Creating Tools

**Goal:** Learn how to create custom tools that agents can use to interact with the outside world.

**What you'll learn:**
- How to use the `@tool` decorator to create tools
- What makes a good tool (name, description, type hints)
- How to call tools directly with `.invoke()`
- Why docstrings are critical for agent tool selection

In [ ]:
from langchain_core.tools import tool

## Step 1: Your First Tool

The `@tool` decorator turns any Python function into a LangChain tool.
The agent reads the function name, docstring, and type hints
to decide **WHEN** and **HOW** to use it.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

print(f"Tool name:        {multiply.name}")
print(f"Tool description: {multiply.description}")
print(f"Tool arguments:   {multiply.args}")

## Step 2: Call It Directly

You can test any tool by calling `.invoke()` with a dict of args.
This is great for debugging before connecting to an agent.

In [ ]:
result = multiply.invoke({"a": 6, "b": 7})
print(f"multiply(6, 7) = {result}")

## Step 3: A Realistic Tool

Real tools interact with data, APIs, or systems.
Here's a tool that looks up UniGPS office information.

In [ ]:
@tool
def get_office_info(city: str) -> str:
    """Get UniGPS office details for a given city. Returns address, team size, and facilities."""
    offices = {
        "bangalore": "WeWork Embassy Tech Village, 5th Floor. 200+ employees. HQ \u2014 all departments.",
        "mumbai": "Worli Business District, Tower A, 12th Floor. 50 employees. Sales & marketing.",
        "hyderabad": "HITEC City, Cyber Gateway, 8th Floor. 80 employees. Engineering hub.",
        "pune": "Hinjewadi Phase 2, Building C, 4th Floor. 40 employees. QA & DevOps.",
    }
    return offices.get(city.lower(), f"No office found in {city}.")

print(f"Tool name: {get_office_info.name}")
print(f"Tool desc: {get_office_info.description}")
print(f"Tool args: {get_office_info.args}")
print(f"\nResult: {get_office_info.invoke({'city': 'Bangalore'})}")
print(f"Result: {get_office_info.invoke({'city': 'Chennai'})}")

## Step 4: Why Docstrings Matter

The docstring is the **MOST** important part of a tool.
Agents read it to decide which tool to use.
Bad docstrings lead to bad agent decisions.

In [ ]:
@tool
def bad_tool(x):
    """Do something."""
    return x * 2

@tool
def good_tool(number: float) -> float:
    """Double a number. Use this when you need to multiply any number by 2."""
    return number * 2

print(f"Bad tool:  name='{bad_tool.name}', desc='{bad_tool.description}'")
print(f"           args={bad_tool.args}")
print(f"Good tool: name='{good_tool.name}', desc='{good_tool.description}'")
print(f"           args={good_tool.args}")
print("\nAgents read the description to decide WHICH tool to use!")
print("Clear descriptions + type hints = better agent decisions.")

## Step 5: Tool with Multiple Parameters

In [ ]:
@tool
def calculate_leave_balance(total_days: int, days_used: int, month: int) -> str:
    """Calculate remaining leave balance for a UniGPS employee.

    Args:
        total_days: Total annual leave allocation
        days_used: Days already taken this year
        month: Current month (1-12) for prorated calculation
    """
    remaining = total_days - days_used
    prorated = round(total_days * month / 12, 1)
    return (f"Used: {days_used}/{total_days} days. "
            f"Remaining: {remaining}. "
            f"Prorated allowance by month {month}: {prorated} days.")

print(f"Args schema: {calculate_leave_balance.args}")
print(calculate_leave_balance.invoke({"total_days": 24, "days_used": 8, "month": 6}))

## TODO 1: Create a Currency Converter Tool

Create a tool called `convert_inr_to_usd` that converts INR to USD.
- Use a fixed exchange rate of **1 USD = 83 INR**
- It should take `amount_inr` (float) as input and return a string with both values, e.g., `"Rs 10,000 = $120.48 USD"`
- Don't forget a clear docstring!

In [ ]:
# @tool
# def convert_inr_to_usd(amount_inr: float) -> str:
#     """___"""
#     usd = ___
#     return f"Rs {amount_inr:,.0f} = ${usd:,.2f} USD"
#
# print(f"Tool: {convert_inr_to_usd.name}")
# print(convert_inr_to_usd.invoke({"amount_inr": 10000}))

## TODO 2: Create a Tech Recommendation Tool

Create a tool called `get_tech_recommendation` that takes a `project_type` (str) parameter
(`"backend"`, `"frontend"`, `"database"`, `"infra"`) and returns UniGPS's recommended technology stack.

Use this data:
- **backend**: `"Python with FastAPI (default) or Java with Spring Boot"`
- **frontend**: `"React with TypeScript (new projects) or Angular (existing apps)"`
- **database**: `"PostgreSQL (relational) or MongoDB (document store)"`
- **infra**: `"AWS EKS (Kubernetes) with Terraform and GitHub Actions CI/CD"`

In [ ]:
# @tool
# def get_tech_recommendation(project_type: str) -> str:
#     """___"""
#     recommendations = {
#         ___
#     }
#     return recommendations.get(project_type.lower(), f"Unknown project type: '{project_type}'")
#
# print(f"Tool: {get_tech_recommendation.name}")
# print(get_tech_recommendation.invoke({"project_type": "backend"}))

## Key Takeaways

- `@tool` decorator turns functions into agent-compatible tools
- Tools have: **name**, **description**, **args** (auto-generated from function)
- Docstrings are critical -- agents read them to pick tools
- Type hints define the input schema for the agent
- `.invoke()` lets you test tools directly before connecting to agents